# CVaR Benchmark for Penalty vs XY-Enforced QAOA

This notebook benchmarks how CVaR (`alpha`) affects:
- expected approximation ratio over **feasible** samples, and
- feasibility probability,

for two enforcement families:
- penalty-enforced QUBO, and
- XY-mixer-enforced QUBO.

It also performs a joint grid search over `(alpha, penalty)` for the penalty-enforced family to maximize feasible-sample expected approximation ratio.

In [1]:
import os
import sys
import json
import warnings
from datetime import datetime
from pathlib import Path

# Repo root (where `max_k_cut/` lives) for imports when cwd is notebooks/experiments/
_REPO_ROOT = Path.cwd().resolve()
for _ in range(6):
    if (_REPO_ROOT / "max_k_cut" / "__init__.py").is_file():
        break
    _REPO_ROOT = _REPO_ROOT.parent
else:
    raise RuntimeError(
        "Could not locate project root containing max_k_cut; "
        "cd to the repo root or notebooks/experiments/ and restart the kernel."
    )
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit_aer.primitives import Sampler as AerSampler
from qiskit.circuit import Parameter
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer

from max_k_cut.models import docplex_QUBO, docplex_QUBO_no_constraints, docplex_RQUBO
from max_k_cut.penalties import interpolated_qubo_penalty, interpolated_rqubo_penalty
from max_k_cut.helpers import (
    generate_graph,
    expected_value,
    feasibility_filter,
    create_dicke_initial_state,
    create_reduced_dicke_initial_state,
    create_ring_xy_mixer,
)
from max_k_cut.qaoa import run_qaoa_extract_samples

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 200,
        "savefig.dpi": 600,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "lines.linewidth": 2.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

np.set_printoptions(precision=4, suppress=True)

In [2]:
# Core experiment settings
SEED = 7
np.random.seed(SEED)

NUM_GRAPHS = 5
NUM_NODES = 5
EDGE_PROBABILITY = 0.5
K = 3
WEIGHTED = False
WEIGHT_RANGE = 1

REPS = 4
ALPHA_GRID = np.array([1.0, 0.75, 0.5, 0.25, 0.10], dtype=float)
PENALTY_GRID = np.linspace(0.0, 1.0, 6)
BASE_SHOTS = 2500
OPTIMIZER_MAXITER = 200

# Baseline reference for summary comparisons
BASELINE_ALPHA = 1.0
TARGET_ALPHA = 0.25
BASELINE_PENALTY = 0.0  # tight side of interpolation

# Output configuration
DATA_DIR = "Data"
os.makedirs(DATA_DIR, exist_ok=True)
RESULT_TAG = f"cvar_penalty_xy_noiseless_p{REPS}_g{NUM_GRAPHS}_n{NUM_NODES}_k{K}"

print("Result tag:", RESULT_TAG)
print("Alpha grid:", ALPHA_GRID)
print("Penalty grid:", PENALTY_GRID)

Result tag: cvar_penalty_xy_noiseless_p4_g5_n5_k3
Alpha grid: [1.   0.75 0.5  0.25 0.1 ]
Penalty grid: [0.  0.2 0.4 0.6 0.8 1. ]


In [3]:
MODEL_COLORS = {
    "penalty_tight": "#1f77b4",
    "penalty_opt": "#9467bd",
    "xy": "#d62728",
    "dicke_rqubo_tight": "#2ca02c",
    "dicke_rqubo_opt": "#17becf",
}


def shots_from_alpha(alpha: float, base_shots: int = BASE_SHOTS) -> int:
    # Keep roughly comparable tail sample count as alpha decreases.
    return int(base_shots / max(alpha, 0.05))


def build_penalty_optimizer(alpha: float, reps: int = REPS, maxiter: int = OPTIMIZER_MAXITER):
    sampler = AerSampler(run_options={"shots": shots_from_alpha(alpha)})
    qaoa_kwargs = {
        "sampler": sampler,
        "optimizer": COBYLA(maxiter=maxiter),
        "reps": reps,
        "initial_point": np.random.rand(2 * reps) * np.pi / 2,
    }
    if alpha < 1.0:
        qaoa_kwargs["aggregation"] = float(alpha)
    qaoa = QAOA(**qaoa_kwargs)
    return MinimumEigenOptimizer(qaoa)


def build_dicke_rqubo_optimizer(num_nodes: int, k: int, alpha: float, reps: int = REPS, maxiter: int = OPTIMIZER_MAXITER):
    # RQUBO penalty model started in the feasible 0-hot/1-hot uniform superposition
    sampler = AerSampler(run_options={"shots": shots_from_alpha(alpha)})
    init_qc = create_reduced_dicke_initial_state(num_nodes, k)
    qaoa_kwargs = {
        "sampler": sampler,
        "optimizer": COBYLA(maxiter=maxiter),
        "reps": reps,
        "initial_state": init_qc,
        "initial_point": np.random.rand(2 * reps) * np.pi / 2,
    }
    if alpha < 1.0:
        qaoa_kwargs["aggregation"] = float(alpha)
    qaoa = QAOA(**qaoa_kwargs)
    return MinimumEigenOptimizer(qaoa)


def build_xy_optimizer(num_nodes: int, k: int, alpha: float, reps: int = REPS, maxiter: int = OPTIMIZER_MAXITER):
    sampler = AerSampler(run_options={"shots": shots_from_alpha(alpha)})
    beta = Parameter("beta")
    init_qc = create_dicke_initial_state(num_nodes, k)
    mixer_qc = create_ring_xy_mixer(num_nodes, k, beta)

    qaoa_kwargs = {
        "sampler": sampler,
        "optimizer": COBYLA(maxiter=maxiter),
        "reps": reps,
        "initial_state": init_qc,
        "mixer": mixer_qc,
        "initial_point": np.random.rand(2 * reps) * np.pi / 2,
    }
    if alpha < 1.0:
        qaoa_kwargs["aggregation"] = float(alpha)
    qaoa = QAOA(**qaoa_kwargs)
    return MinimumEigenOptimizer(qaoa)


def run_and_score_feasible(graph, model, label, optimizer):
    results = run_qaoa_extract_samples([model], [label], optimizer)
    samples = results[label]["samples"]
    feasible_samples, feasibility_prob = feasibility_filter(graph, K, samples, label=label)
    exp_ratio_feasible = expected_value(feasible_samples) if len(feasible_samples) > 0 else np.nan
    return exp_ratio_feasible, feasibility_prob


def assert_valid_metric_array(arr: np.ndarray, name: str):
    if np.isnan(arr).any():
        nan_count = np.isnan(arr).sum()
        raise ValueError(f"{name} contains {nan_count} NaN entries.")
    if np.isinf(arr).any():
        inf_count = np.isinf(arr).sum()
        raise ValueError(f"{name} contains {inf_count} inf entries.")


def summarize_over_graphs(arr: np.ndarray):
    return np.mean(arr, axis=0), np.std(arr, axis=0)

In [4]:
graphs = [
    generate_graph(
        NUM_NODES,
        EDGE_PROBABILITY,
        weighted=WEIGHTED,
        weight_range=WEIGHT_RANGE,
        seed=SEED + idx,
    )
    for idx in range(NUM_GRAPHS)
]

print(f"Generated {len(graphs)} graphs.")

Generated 5 graphs.


In [ ]:
penalty_exp = np.zeros((NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID)))
penalty_feas = np.zeros((NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID)))
xy_exp = np.zeros((NUM_GRAPHS, len(ALPHA_GRID)))
xy_feas = np.zeros((NUM_GRAPHS, len(ALPHA_GRID)))
dicke_rqubo_exp = np.zeros((NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID)))
dicke_rqubo_feas = np.zeros((NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID)))

for g_idx, graph in enumerate(graphs):
    print(f"\n=== Graph {g_idx + 1}/{NUM_GRAPHS} ===")

    for a_idx, alpha in enumerate(ALPHA_GRID):
        print(f"  Alpha = {alpha:.2f}")

        # XY-enforced run (no penalties in objective)
        xy_model = docplex_QUBO_no_constraints(graph, K, name=f"XY_G{g_idx}_A{alpha:.2f}")
        xy_optimizer = build_xy_optimizer(NUM_NODES, K, alpha)
        xy_exp_val, xy_feas_prob = run_and_score_feasible(
            graph=graph,
            model=xy_model,
            label="QUBO_XY",
            optimizer=xy_optimizer,
        )
        xy_exp[g_idx, a_idx] = xy_exp_val
        xy_feas[g_idx, a_idx] = xy_feas_prob

        # Penalty-enforced sweep across interpolation parameter
        penalty_optimizer = build_penalty_optimizer(alpha)
        for p_idx, pen_interp in enumerate(PENALTY_GRID):
            penalties = interpolated_qubo_penalty(graph, K, pen_interp)
            penalty_model = docplex_QUBO(
                graph,
                K,
                penalty=penalties,
                name=f"PEN_G{g_idx}_A{alpha:.2f}_P{pen_interp:.2f}",
            )
            exp_val, feas_prob = run_and_score_feasible(
                graph=graph,
                model=penalty_model,
                label="QUBO_PEN",
                optimizer=penalty_optimizer,
            )
            penalty_exp[g_idx, a_idx, p_idx] = exp_val
            penalty_feas[g_idx, a_idx, p_idx] = feas_prob

        # Dicke-RQUBO sweep: reduced encoding, feasible (0-hot/1-hot) initial state
        dicke_rqubo_optimizer = build_dicke_rqubo_optimizer(NUM_NODES, K, alpha)
        for p_idx, pen_interp in enumerate(PENALTY_GRID):
            penalties_r = interpolated_rqubo_penalty(graph, K, pen_interp)
            dicke_rqubo_model = docplex_RQUBO(
                graph,
                K,
                penalty=penalties_r,
                name=f"DRQ_G{g_idx}_A{alpha:.2f}_P{pen_interp:.2f}",
            )
            exp_val, feas_prob = run_and_score_feasible(
                graph=graph,
                model=dicke_rqubo_model,
                label="RQUBO_DICKE",
                optimizer=dicke_rqubo_optimizer,
            )
            dicke_rqubo_exp[g_idx, a_idx, p_idx] = exp_val
            dicke_rqubo_feas[g_idx, a_idx, p_idx] = feas_prob

print("\nAll sweeps complete.")


=== Graph 1/5 ===
  Alpha = 1.00
Solving QUBO_XY:
Classically calculating max objective value...
Max objective value for QUBO_XY: 7.0
Running QAOA...
QAOA result: objective function value: 7.0
variable values: x00=0.0, x01=0.0, x02=1.0, x10=1.0, x11=0.0, x12=0.0, x20=1.0, x21=0.0, x22=0.0, x30=0.0, x31=0.0, x32=1.0, x40=0.0, x41=1.0, x42=0.0
status: SUCCESS
Time taken: 2222.4591 seconds
Probability of feasible solution for QUBO_XY: 1.0000
Solving QUBO_PEN:
Classically calculating max objective value...
Max objective value for QUBO_PEN: 7.000000000000001
Running QAOA...
QAOA result: objective function value: 7.0
variable values: x00=0.0, x01=0.0, x02=1.0, x10=1.0, x11=0.0, x12=0.0, x20=1.0, x21=0.0, x22=0.0, x30=0.0, x31=0.0, x32=1.0, x40=0.0, x41=1.0, x42=0.0
status: SUCCESS
Time taken: 44.5630 seconds
Probability of feasible solution for QUBO_PEN: 0.0452
Solving QUBO_PEN:
Classically calculating max objective value...
Max objective value for QUBO_PEN: 7.0
Running QAOA...
QAOA result:

In [ ]:
assert penalty_exp.shape == (NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID))
assert penalty_feas.shape == (NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID))
assert xy_exp.shape == (NUM_GRAPHS, len(ALPHA_GRID))
assert xy_feas.shape == (NUM_GRAPHS, len(ALPHA_GRID))
assert dicke_rqubo_exp.shape == (NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID))
assert dicke_rqubo_feas.shape == (NUM_GRAPHS, len(ALPHA_GRID), len(PENALTY_GRID))

assert_valid_metric_array(penalty_exp, "penalty_exp")
assert_valid_metric_array(penalty_feas, "penalty_feas")
assert_valid_metric_array(xy_exp, "xy_exp")
assert_valid_metric_array(xy_feas, "xy_feas")
assert_valid_metric_array(dicke_rqubo_exp, "dicke_rqubo_exp")
assert_valid_metric_array(dicke_rqubo_feas, "dicke_rqubo_feas")

# Means/stds for fixed tight-penalty baseline
baseline_penalty_idx = int(np.argmin(np.abs(PENALTY_GRID - BASELINE_PENALTY)))
penalty_tight_exp = penalty_exp[:, :, baseline_penalty_idx]
penalty_tight_feas = penalty_feas[:, :, baseline_penalty_idx]

penalty_tight_exp_mean, penalty_tight_exp_std = summarize_over_graphs(penalty_tight_exp)
penalty_tight_feas_mean, penalty_tight_feas_std = summarize_over_graphs(penalty_tight_feas)

dicke_rqubo_tight_exp = dicke_rqubo_exp[:, :, baseline_penalty_idx]
dicke_rqubo_tight_feas = dicke_rqubo_feas[:, :, baseline_penalty_idx]
dicke_rqubo_tight_exp_mean, dicke_rqubo_tight_exp_std = summarize_over_graphs(dicke_rqubo_tight_exp)
dicke_rqubo_tight_feas_mean, dicke_rqubo_tight_feas_std = summarize_over_graphs(dicke_rqubo_tight_feas)
xy_exp_mean, xy_exp_std = summarize_over_graphs(xy_exp)
xy_feas_mean, xy_feas_std = summarize_over_graphs(xy_feas)

# Joint optimization over (alpha, penalty) for penalty-enforced model
mean_penalty_exp_surface = np.mean(penalty_exp, axis=0)
std_penalty_exp_surface = np.std(penalty_exp, axis=0)
mean_penalty_feas_surface = np.mean(penalty_feas, axis=0)
std_penalty_feas_surface = np.std(penalty_feas, axis=0)

best_flat_idx = int(np.argmax(mean_penalty_exp_surface))
best_alpha_idx, best_penalty_idx = np.unravel_index(best_flat_idx, mean_penalty_exp_surface.shape)

best_alpha = float(ALPHA_GRID[best_alpha_idx])
best_penalty = float(PENALTY_GRID[best_penalty_idx])
best_mean_exp = float(mean_penalty_exp_surface[best_alpha_idx, best_penalty_idx])
best_mean_feas = float(mean_penalty_feas_surface[best_alpha_idx, best_penalty_idx])

# Best penalty for each alpha (curve used in main figure)
best_penalty_idx_per_alpha = np.argmax(mean_penalty_exp_surface, axis=1)
penalty_opt_exp_mean = mean_penalty_exp_surface[np.arange(len(ALPHA_GRID)), best_penalty_idx_per_alpha]
penalty_opt_feas_mean = mean_penalty_feas_surface[np.arange(len(ALPHA_GRID)), best_penalty_idx_per_alpha]

# Compute std across graphs at each alpha using per-alpha best penalty index
penalty_opt_exp_across_graphs = np.zeros((NUM_GRAPHS, len(ALPHA_GRID)))
penalty_opt_feas_across_graphs = np.zeros((NUM_GRAPHS, len(ALPHA_GRID)))
for a_idx in range(len(ALPHA_GRID)):
    p_idx = int(best_penalty_idx_per_alpha[a_idx])
    penalty_opt_exp_across_graphs[:, a_idx] = penalty_exp[:, a_idx, p_idx]
    penalty_opt_feas_across_graphs[:, a_idx] = penalty_feas[:, a_idx, p_idx]

penalty_opt_exp_std = np.std(penalty_opt_exp_across_graphs, axis=0)
penalty_opt_feas_std = np.std(penalty_opt_feas_across_graphs, axis=0)

# Dicke-RQUBO: mean surfaces and per-alpha best penalty (mirrors the penalty model)
mean_dicke_rqubo_exp_surface = np.mean(dicke_rqubo_exp, axis=0)
mean_dicke_rqubo_feas_surface = np.mean(dicke_rqubo_feas, axis=0)
dicke_rqubo_best_penalty_idx_per_alpha = np.argmax(mean_dicke_rqubo_exp_surface, axis=1)
dicke_rqubo_opt_exp_mean = mean_dicke_rqubo_exp_surface[np.arange(len(ALPHA_GRID)), dicke_rqubo_best_penalty_idx_per_alpha]
dicke_rqubo_opt_feas_mean = mean_dicke_rqubo_feas_surface[np.arange(len(ALPHA_GRID)), dicke_rqubo_best_penalty_idx_per_alpha]

dicke_rqubo_opt_exp_across_graphs = np.zeros((NUM_GRAPHS, len(ALPHA_GRID)))
dicke_rqubo_opt_feas_across_graphs = np.zeros((NUM_GRAPHS, len(ALPHA_GRID)))
for a_idx in range(len(ALPHA_GRID)):
    p_idx = int(dicke_rqubo_best_penalty_idx_per_alpha[a_idx])
    dicke_rqubo_opt_exp_across_graphs[:, a_idx] = dicke_rqubo_exp[:, a_idx, p_idx]
    dicke_rqubo_opt_feas_across_graphs[:, a_idx] = dicke_rqubo_feas[:, a_idx, p_idx]

dicke_rqubo_opt_exp_std = np.std(dicke_rqubo_opt_exp_across_graphs, axis=0)
dicke_rqubo_opt_feas_std = np.std(dicke_rqubo_opt_feas_across_graphs, axis=0)

# Per-graph optimal (alpha, penalty) for robustness diagnostics
per_graph_best = []
for g_idx in range(NUM_GRAPHS):
    g_flat = int(np.argmax(penalty_exp[g_idx]))
    g_a_idx, g_p_idx = np.unravel_index(g_flat, penalty_exp[g_idx].shape)
    per_graph_best.append(
        {
            "graph_index": int(g_idx),
            "alpha": float(ALPHA_GRID[g_a_idx]),
            "penalty": float(PENALTY_GRID[g_p_idx]),
            "exp_ratio_feasible": float(penalty_exp[g_idx, g_a_idx, g_p_idx]),
            "feasibility_prob": float(penalty_feas[g_idx, g_a_idx, g_p_idx]),
        }
    )

per_graph_best_df = pd.DataFrame(per_graph_best)
per_graph_best_df.head()

In [ ]:
fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(8.2, 7.2), sharex=True)

# Top panel: feasible expected approximation ratio
ax_top.plot(ALPHA_GRID, penalty_tight_exp_mean, marker="o", color=MODEL_COLORS["penalty_tight"], label="Penalty QUBO (tight)")
ax_top.fill_between(
    ALPHA_GRID,
    penalty_tight_exp_mean - penalty_tight_exp_std,
    penalty_tight_exp_mean + penalty_tight_exp_std,
    color=MODEL_COLORS["penalty_tight"],
    alpha=0.18,
)

ax_top.plot(ALPHA_GRID, penalty_opt_exp_mean, marker="s", color=MODEL_COLORS["penalty_opt"], label="Penalty QUBO (penalty-optimized)")
ax_top.fill_between(
    ALPHA_GRID,
    penalty_opt_exp_mean - penalty_opt_exp_std,
    penalty_opt_exp_mean + penalty_opt_exp_std,
    color=MODEL_COLORS["penalty_opt"],
    alpha=0.15,
)

ax_top.plot(ALPHA_GRID, xy_exp_mean, marker="^", color=MODEL_COLORS["xy"], label="XY mixer enforced")
ax_top.fill_between(
    ALPHA_GRID,
    xy_exp_mean - xy_exp_std,
    xy_exp_mean + xy_exp_std,
    color=MODEL_COLORS["xy"],
    alpha=0.18,
)

ax_top.plot(ALPHA_GRID, dicke_rqubo_tight_exp_mean, marker="D", color=MODEL_COLORS["dicke_rqubo_tight"], label="Dicke RQUBO (tight)")
ax_top.fill_between(
    ALPHA_GRID,
    dicke_rqubo_tight_exp_mean - dicke_rqubo_tight_exp_std,
    dicke_rqubo_tight_exp_mean + dicke_rqubo_tight_exp_std,
    color=MODEL_COLORS["dicke_rqubo_tight"],
    alpha=0.18,
)

ax_top.plot(ALPHA_GRID, dicke_rqubo_opt_exp_mean, marker="v", color=MODEL_COLORS["dicke_rqubo_opt"], label="Dicke RQUBO (penalty-optimized)")
ax_top.fill_between(
    ALPHA_GRID,
    dicke_rqubo_opt_exp_mean - dicke_rqubo_opt_exp_std,
    dicke_rqubo_opt_exp_mean + dicke_rqubo_opt_exp_std,
    color=MODEL_COLORS["dicke_rqubo_opt"],
    alpha=0.15,
)

ax_top.set_ylabel("Expected ratio | feasible")
ax_top.set_title("CVaR impact differs for penalty vs XY enforcement")
ax_top.grid(alpha=0.25, linestyle="--")
ax_top.legend(loc="best", ncol=1)

# Bottom panel: feasibility probability
ax_bottom.plot(ALPHA_GRID, penalty_tight_feas_mean, marker="o", color=MODEL_COLORS["penalty_tight"], label="Penalty QUBO (tight)")
ax_bottom.fill_between(
    ALPHA_GRID,
    penalty_tight_feas_mean - penalty_tight_feas_std,
    penalty_tight_feas_mean + penalty_tight_feas_std,
    color=MODEL_COLORS["penalty_tight"],
    alpha=0.18,
)

ax_bottom.plot(ALPHA_GRID, penalty_opt_feas_mean, marker="s", color=MODEL_COLORS["penalty_opt"], label="Penalty QUBO (penalty-optimized)")
ax_bottom.fill_between(
    ALPHA_GRID,
    penalty_opt_feas_mean - penalty_opt_feas_std,
    penalty_opt_feas_mean + penalty_opt_feas_std,
    color=MODEL_COLORS["penalty_opt"],
    alpha=0.15,
)

ax_bottom.plot(ALPHA_GRID, xy_feas_mean, marker="^", color=MODEL_COLORS["xy"], label="XY mixer enforced")
ax_bottom.fill_between(
    ALPHA_GRID,
    xy_feas_mean - xy_feas_std,
    xy_feas_mean + xy_feas_std,
    color=MODEL_COLORS["xy"],
    alpha=0.18,
)

ax_bottom.plot(ALPHA_GRID, dicke_rqubo_tight_feas_mean, marker="D", color=MODEL_COLORS["dicke_rqubo_tight"], label="Dicke RQUBO (tight)")
ax_bottom.fill_between(
    ALPHA_GRID,
    dicke_rqubo_tight_feas_mean - dicke_rqubo_tight_feas_std,
    dicke_rqubo_tight_feas_mean + dicke_rqubo_tight_feas_std,
    color=MODEL_COLORS["dicke_rqubo_tight"],
    alpha=0.18,
)

ax_bottom.plot(ALPHA_GRID, dicke_rqubo_opt_feas_mean, marker="v", color=MODEL_COLORS["dicke_rqubo_opt"], label="Dicke RQUBO (penalty-optimized)")
ax_bottom.fill_between(
    ALPHA_GRID,
    dicke_rqubo_opt_feas_mean - dicke_rqubo_opt_feas_std,
    dicke_rqubo_opt_feas_mean + dicke_rqubo_opt_feas_std,
    color=MODEL_COLORS["dicke_rqubo_opt"],
    alpha=0.15,
)

ax_bottom.set_xlabel("CVaR alpha")
ax_bottom.set_ylabel("Feasibility probability")
ax_bottom.grid(alpha=0.25, linestyle="--")

fig.tight_layout()
comparison_pdf_path = os.path.join(DATA_DIR, f"cvar_effect_comparison_{RESULT_TAG}.pdf")
fig.savefig(comparison_pdf_path, bbox_inches="tight")
print("Saved:", comparison_pdf_path)
plt.show()

In [ ]:
A, P = np.meshgrid(ALPHA_GRID, PENALTY_GRID, indexing="ij")

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)

im0 = axes[0].pcolormesh(P, A, mean_penalty_exp_surface, shading="nearest", cmap="viridis")
axes[0].scatter([best_penalty], [best_alpha], c="white", edgecolor="black", s=70, marker="*", label="Global optimum")
axes[0].set_title("Penalty QUBO: mean expected ratio | feasible")
axes[0].set_xlabel("Penalty interpolation")
axes[0].set_ylabel("CVaR alpha")
axes[0].legend(loc="lower right")
cb0 = fig.colorbar(im0, ax=axes[0])
cb0.set_label("Mean expected ratio | feasible")

im1 = axes[1].pcolormesh(P, A, mean_penalty_feas_surface, shading="nearest", cmap="magma")
axes[1].set_title("Penalty QUBO: mean feasibility probability")
axes[1].set_xlabel("Penalty interpolation")
axes[1].set_ylabel("CVaR alpha")
cb1 = fig.colorbar(im1, ax=axes[1])
cb1.set_label("Mean feasibility probability")

heatmap_pdf_path = os.path.join(DATA_DIR, f"cvar_penalty_heatmaps_{RESULT_TAG}.pdf")
fig.savefig(heatmap_pdf_path, bbox_inches="tight")
print("Saved:", heatmap_pdf_path)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)

im0 = axes[0].pcolormesh(P, A, mean_dicke_rqubo_exp_surface, shading="nearest", cmap="viridis")
axes[0].set_title("Dicke RQUBO: mean expected ratio | feasible")
axes[0].set_xlabel("Penalty interpolation")
axes[0].set_ylabel("CVaR alpha")
cb0 = fig.colorbar(im0, ax=axes[0])
cb0.set_label("Mean expected ratio | feasible")

im1 = axes[1].pcolormesh(P, A, mean_dicke_rqubo_feas_surface, shading="nearest", cmap="magma")
axes[1].set_title("Dicke RQUBO: mean feasibility probability")
axes[1].set_xlabel("Penalty interpolation")
axes[1].set_ylabel("CVaR alpha")
cb1 = fig.colorbar(im1, ax=axes[1])
cb1.set_label("Mean feasibility probability")

dicke_rqubo_heatmap_pdf_path = os.path.join(DATA_DIR, f"cvar_dicke_rqubo_heatmaps_{RESULT_TAG}.pdf")
fig.savefig(dicke_rqubo_heatmap_pdf_path, bbox_inches="tight")
print("Saved:", dicke_rqubo_heatmap_pdf_path)
plt.show()


In [ ]:
def idx_for_alpha(alpha):
    idx = int(np.argmin(np.abs(ALPHA_GRID - alpha)))
    if not np.isclose(ALPHA_GRID[idx], alpha):
        raise ValueError(f"Alpha {alpha} not in ALPHA_GRID")
    return idx


base_a_idx = idx_for_alpha(BASELINE_ALPHA)
target_a_idx = idx_for_alpha(TARGET_ALPHA)

penalty_tight_delta = float(penalty_tight_exp_mean[target_a_idx] - penalty_tight_exp_mean[base_a_idx])
penalty_opt_delta = float(penalty_opt_exp_mean[target_a_idx] - penalty_opt_exp_mean[base_a_idx])
xy_delta = float(xy_exp_mean[target_a_idx] - xy_exp_mean[base_a_idx])

summary_df = pd.DataFrame(
    [
        {
            "model": "Penalty QUBO (tight)",
            "exp_ratio_alpha_1.0": float(penalty_tight_exp_mean[base_a_idx]),
            "exp_ratio_alpha_0.25": float(penalty_tight_exp_mean[target_a_idx]),
            "delta_0.25_minus_1.0": penalty_tight_delta,
        },
        {
            "model": "Penalty QUBO (penalty-optimized)",
            "exp_ratio_alpha_1.0": float(penalty_opt_exp_mean[base_a_idx]),
            "exp_ratio_alpha_0.25": float(penalty_opt_exp_mean[target_a_idx]),
            "delta_0.25_minus_1.0": penalty_opt_delta,
        },
        {
            "model": "XY mixer enforced",
            "exp_ratio_alpha_1.0": float(xy_exp_mean[base_a_idx]),
            "exp_ratio_alpha_0.25": float(xy_exp_mean[target_a_idx]),
            "delta_0.25_minus_1.0": xy_delta,
        },
        {
            "model": "Dicke RQUBO (tight)",
            "exp_ratio_alpha_1.0": float(dicke_rqubo_tight_exp_mean[base_a_idx]),
            "exp_ratio_alpha_0.25": float(dicke_rqubo_tight_exp_mean[target_a_idx]),
            "delta_0.25_minus_1.0": float(
                dicke_rqubo_tight_exp_mean[target_a_idx] - dicke_rqubo_tight_exp_mean[base_a_idx]
            ),
        },
        {
            "model": "Dicke RQUBO (penalty-optimized)",
            "exp_ratio_alpha_1.0": float(dicke_rqubo_opt_exp_mean[base_a_idx]),
            "exp_ratio_alpha_0.25": float(dicke_rqubo_opt_exp_mean[target_a_idx]),
            "delta_0.25_minus_1.0": float(
                dicke_rqubo_opt_exp_mean[target_a_idx] - dicke_rqubo_opt_exp_mean[base_a_idx]
            ),
        },
        {
            "model": "Penalty QUBO global optimum",
            "exp_ratio_alpha_1.0": float(mean_penalty_exp_surface[base_a_idx, baseline_penalty_idx]),
            "exp_ratio_alpha_0.25": float(
                mean_penalty_exp_surface[target_a_idx, best_penalty_idx_per_alpha[target_a_idx]]
            ),
            "delta_0.25_minus_1.0": float(
                mean_penalty_exp_surface[target_a_idx, best_penalty_idx_per_alpha[target_a_idx]]
                - mean_penalty_exp_surface[base_a_idx, baseline_penalty_idx]
            ),
        },
    ]
)

print("Global best (penalty model):")
print(f"  alpha* = {best_alpha:.2f}, penalty* = {best_penalty:.2f}")
print(f"  mean expected ratio|feasible = {best_mean_exp:.5f}")
print(f"  mean feasibility probability = {best_mean_feas:.5f}")
print()
display(summary_df)

print("Per-graph best (first 10 rows):")
display(per_graph_best_df.head(10))

In [ ]:
results_npz_path = os.path.join(DATA_DIR, f"{RESULT_TAG}.npz")
meta_json_path = os.path.join(DATA_DIR, f"{RESULT_TAG}_metadata.json")

np.savez(
    results_npz_path,
    alpha_grid=ALPHA_GRID,
    penalty_grid=PENALTY_GRID,
    penalty_exp=penalty_exp,
    penalty_feas=penalty_feas,
    xy_exp=xy_exp,
    xy_feas=xy_feas,
    mean_penalty_exp_surface=mean_penalty_exp_surface,
    std_penalty_exp_surface=std_penalty_exp_surface,
    mean_penalty_feas_surface=mean_penalty_feas_surface,
    std_penalty_feas_surface=std_penalty_feas_surface,
    penalty_tight_exp_mean=penalty_tight_exp_mean,
    penalty_tight_exp_std=penalty_tight_exp_std,
    penalty_tight_feas_mean=penalty_tight_feas_mean,
    penalty_tight_feas_std=penalty_tight_feas_std,
    penalty_opt_exp_mean=penalty_opt_exp_mean,
    penalty_opt_exp_std=penalty_opt_exp_std,
    penalty_opt_feas_mean=penalty_opt_feas_mean,
    penalty_opt_feas_std=penalty_opt_feas_std,
    xy_exp_mean=xy_exp_mean,
    xy_exp_std=xy_exp_std,
    xy_feas_mean=xy_feas_mean,
    xy_feas_std=xy_feas_std,
    dicke_rqubo_exp=dicke_rqubo_exp,
    dicke_rqubo_feas=dicke_rqubo_feas,
    mean_dicke_rqubo_exp_surface=mean_dicke_rqubo_exp_surface,
    mean_dicke_rqubo_feas_surface=mean_dicke_rqubo_feas_surface,
    dicke_rqubo_tight_exp_mean=dicke_rqubo_tight_exp_mean,
    dicke_rqubo_tight_exp_std=dicke_rqubo_tight_exp_std,
    dicke_rqubo_tight_feas_mean=dicke_rqubo_tight_feas_mean,
    dicke_rqubo_tight_feas_std=dicke_rqubo_tight_feas_std,
    dicke_rqubo_opt_exp_mean=dicke_rqubo_opt_exp_mean,
    dicke_rqubo_opt_exp_std=dicke_rqubo_opt_exp_std,
    dicke_rqubo_opt_feas_mean=dicke_rqubo_opt_feas_mean,
    dicke_rqubo_opt_feas_std=dicke_rqubo_opt_feas_std,
    dicke_rqubo_best_penalty_idx_per_alpha=dicke_rqubo_best_penalty_idx_per_alpha,
    best_penalty_idx_per_alpha=best_penalty_idx_per_alpha,
    best_alpha=best_alpha,
    best_penalty=best_penalty,
    best_mean_exp=best_mean_exp,
    best_mean_feas=best_mean_feas,
)

metadata = {
    "result_tag": RESULT_TAG,
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    "seed": SEED,
    "num_graphs": NUM_GRAPHS,
    "num_nodes": NUM_NODES,
    "edge_probability": EDGE_PROBABILITY,
    "k": K,
    "weighted": WEIGHTED,
    "weight_range": WEIGHT_RANGE,
    "reps": REPS,
    "alpha_grid": ALPHA_GRID.tolist(),
    "penalty_grid": PENALTY_GRID.tolist(),
    "base_shots": BASE_SHOTS,
    "optimizer": "COBYLA",
    "optimizer_maxiter": OPTIMIZER_MAXITER,
    "noisy": False,
    "best_global": {
        "alpha": best_alpha,
        "penalty": best_penalty,
        "mean_expected_ratio_feasible": best_mean_exp,
        "mean_feasibility_probability": best_mean_feas,
    },
    "paths": {
        "results_npz": results_npz_path,
        "comparison_pdf": comparison_pdf_path,
        "heatmap_pdf": heatmap_pdf_path,
    },
}

with open(meta_json_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", results_npz_path)
print("Saved:", meta_json_path)

In [ ]:
# Optional diagnostic cell
RUN_CLASSICAL_DIAGNOSTIC = False

if RUN_CLASSICAL_DIAGNOSTIC:
    g = graphs[0]
    penalties = interpolated_qubo_penalty(g, K, PENALTY_GRID[0])
    model = docplex_QUBO(g, K, penalties, name="diagnostic_qubo")
    classical = MinimumEigenOptimizer(NumPyMinimumEigensolver())
    diag_exp, diag_feas = run_and_score_feasible(g, model, "QUBO_DIAG", classical)
    print("Classical diagnostic expected ratio|feasible:", diag_exp)
    print("Classical diagnostic feasibility:", diag_feas)
else:
    print("Set RUN_CLASSICAL_DIAGNOSTIC=True to execute diagnostic.")